In [ ]:

#!/usr/bin/env python3
"""PCIR unified reproduction: Table 3 (DIMACS) + Table 4 (canonical ablation).

Run in Colab or Jupyter:
    %run /content/pcir_table3_table4.py

Reads DIMACS from /content/drive/MyDrive/GPU/DIMACS_all_ascii
Writes CSV, Markdown, and JSON to <DIMACS_DIR>/pcir_reproduction_outputs/

What this script does
---------------------
Table 3: runs the pruned Algorithm 2 on each configured DIMACS instance with a
30 s cutoff. Corrects the notebook's original defects:
  * queen6_6.col target corrected from 6 to 7 (OEIS A088202);
  * benchmark grouping is explicit and non-overlapping (le450_5a and le450_5b
    no longer run twice);
  * status separates target_reached, timed_out, search_exhausted;
  * DIMACS parser accepts both header conventions (undirected == parsed,
    or doubled == 2 * parsed, as used by the queen files).

Table 4: two-row canonical-ordering ablation on myciel3.col and myciel4.col.
For each instance, reports the popped-occurrence count for the canonical and
noncanonical candidate rule, run to exhaustion with timeout disabled.
  * myciel3: both the notebook engine and an independent memoised counter
    reproduce 2,325 / 969,712 (the paper's 5,394 is not reproduced).
  * myciel4: both methods reproduce 929,315 canonical. The noncanonical count
    of 3,450,514,238,574,088 is obtained by the memoised counter only; the
    direct engine cannot materialise that many states in memory, so it is
    skipped for that one case. The two methods agree on every case that both
    can compute, which validates the counter for the larger case.

The three notebook functions (is_maximal, calculate_candidate_set,
run_pure_downward_engine) are reproduced verbatim so the Table 4 ablation
matches Jim's reproduction exactly.
"""
import csv, hashlib, json, platform, time
from datetime import datetime, timezone
from functools import lru_cache
from pathlib import Path

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
DIMACS_DIR = Path('/content/drive/MyDrive/GPU/DIMACS_all_ascii')
OUT_DIR = DIMACS_DIR / 'pcir_reproduction_outputs'
TIMEOUT_SECONDS = 30.0

# Corrected target dictionary.
DIMACS_TARGETS = {
    'myciel3.col': 4,
    'myciel4.col': 5,
    'queen5_5.col': 5,
    'myciel5.col': 6,
    'queen6_6.col': 7,       # corrected: was 6
    'queen7_7.col': 7,
    'le450_5a.col': 5,
    'le450_5b.col': 5,
    'myciel6.col': 7,
    'myciel7.col': 8,
    'queen8_8.col': 9,
    'queen9_9.col': 10,
    'le450_15a.col': 15,
    'le450_25a.col': 25,
}

# Explicit, non-overlapping benchmark groups.
SMALL_NAMES  = {'myciel3.col', 'myciel4.col', 'queen5_5.col'}
MEDIUM_NAMES = {'myciel5.col', 'queen6_6.col', 'queen7_7.col',
                'le450_5a.col', 'le450_5b.col'}
LARGE_NAMES  = {'myciel6.col', 'myciel7.col', 'queen8_8.col', 'queen9_9.col',
                'le450_15a.col', 'le450_25a.col'}

# Table 4 instances. myciel3 is embedded; myciel4 is read from DIMACS_DIR.
MYCIEL3_TEXT = '''c FILE: myciel3.col
p edge 11 20
e 1 2
e 1 4
e 1 7
e 1 9
e 2 3
e 2 6
e 2 8
e 3 5
e 3 7
e 3 10
e 4 5
e 4 6
e 4 10
e 5 8
e 5 9
e 6 11
e 7 11
e 8 11
e 9 11
e 10 11
'''
TABLE4_PAPER    = {
    ('myciel3.col', 'canonical'):    2325,
    ('myciel3.col', 'noncanonical'): 5394,
    ('myciel4.col', 'canonical'):    929315,      # paper Table 3 row
    ('myciel4.col', 'noncanonical'): 2774074,     # paper Table 4 row
}
TABLE4_EXPECTED = {
    ('myciel3.col', 'canonical'):    2325,
    ('myciel3.col', 'noncanonical'): 969712,
    ('myciel4.col', 'canonical'):    929315,
    ('myciel4.col', 'noncanonical'): 3450514238574088,
}
# The noncanonical myciel4 count is ~3.45e15; the direct engine cannot
# materialise that many states, so it is skipped for that one case.
SKIP_ENGINE = {('myciel4.col', 'noncanonical')}

GRAPH_URL          = 'https://mat.tepper.cmu.edu/COLOR/instances/myciel3.col'
NOTEBOOK_GIT_BLOB  = '8e1ba825363fddd146af4af601ba217a83e23510'

# ---------------------------------------------------------------------------
# DIMACS parsing (accepts undirected OR doubled header convention)
# ---------------------------------------------------------------------------
def parse_dimacs_text(text):
    n = m = None
    undirected = set()
    for raw in text.splitlines():
        line = raw.strip()
        if not line or line.startswith('c'):
            continue
        f = line.split()
        if len(f) == 4 and f[:2] == ['p', 'edge']:
            n, m = int(f[2]), int(f[3])
        elif len(f) == 3 and f[0] == 'e':
            u, v = int(f[1]) - 1, int(f[2]) - 1
            if u == v:
                raise ValueError('Self-loop in input')
            undirected.add(tuple(sorted((u, v))))
        else:
            raise ValueError('Unrecognized DIMACS line: ' + line)
    if n is None or m is None:
        raise ValueError('Missing DIMACS header')
    if any(u < 0 or v >= n for u, v in undirected):
        raise ValueError('Vertex out of range')
    parsed = len(undirected)
    if parsed == m:
        convention = 'undirected'
    elif 2 * parsed == m:
        convention = 'doubled'
    else:
        raise ValueError(
            f'Header says {m} edges; parsed {parsed} undirected '
            f'({2 * parsed} directed). Neither convention matches.')
    edges = undirected | {(v, u) for u, v in undirected}
    return list(range(n)), edges, convention, m, parsed

def parse_file(path):
    """Returns (V, E, sha256_hex, convention, header_m, parsed_undirected_m)."""
    raw = path.read_bytes()
    V, E, conv, hm, pm = parse_dimacs_text(raw.decode('utf-8'))
    return V, E, hashlib.sha256(raw).hexdigest(), conv, hm, pm

# ---------------------------------------------------------------------------
# Engine (verbatim notebook functions, with a canonical/noncanonical switch)
# ---------------------------------------------------------------------------
def is_maximal(I, P, edges):
    """Pure Downward relative maximality check over active pool P."""
    remaining = set(P) - set(I)
    for v in remaining:
        if all((v, u) not in edges for u in I):
            return False
    return True

def candidate_fn(canonical):
    def candidates(P, I, E):
        if not I:
            return tuple(sorted(P))
        mx = max(I)
        return tuple(sorted(
            v for v in P if v not in I
            and (not canonical or v > mx)
            and all((u, v) not in E for u in I)))
    return candidates

def run_engine(V, E, target, timeout=30.0, canonical=True):
    """Notebook-equivalent DFS engine with unambiguous status flags."""
    start  = time.perf_counter()
    cand   = candidate_fn(canonical)
    P0     = tuple(sorted(V))
    frontier = [(P0, (), (), cand(P0, (), E))]
    best   = len(V) + 1
    popped = 0
    timed_out = False
    while frontier:
        if time.perf_counter() - start >= timeout:
            timed_out = True
            break
        p, r, i, c = frontier.pop()
        popped += 1
        if len(r) > best:
            continue
        if not p:
            if len(r) < best:
                best = len(r)
            continue
        if c:
            for v in sorted(c, reverse=True):
                ni = tuple(sorted((*i, v)))
                frontier.append((p, r, ni, cand(p, ni, E)))
        elif i and is_maximal(i, p, E):
            np = tuple(x for x in p if x not in i)
            frontier.append((np, r + (i,), (), cand(np, (), E)))
    elapsed   = time.perf_counter() - start
    exhausted = (not frontier) and (not timed_out)
    best      = best if best <= len(V) else None
    reached   = (target is not None) and (best == target)
    if exhausted and reached:
        status = 'TARGET_REACHED_EXHAUSTIVE'
    elif exhausted:
        status = 'EXHAUSTED_TARGET_NOT_REACHED'
    elif reached:
        status = 'TARGET_REACHED_TIMEOUT'
    else:
        status = 'TIMEOUT_TARGET_NOT_REACHED'
    return {
        'best_found': best,
        'target_chi': target,
        'target_reached': reached,
        'explored_popped_occurrences': popped,
        'runtime_seconds': round(elapsed, 6),
        'timed_out': timed_out,
        'search_exhausted': exhausted,
        'status': status,
    }

# ---------------------------------------------------------------------------
# Independent memoised counter (same as Jim's script)
# ---------------------------------------------------------------------------
def independent_count(V, E, canonical):
    n = len(V)
    adj = [sum(1 << v for v in V if (u, v) in E) for u in V]
    @lru_cache(maxsize=None)
    def visit(P, k, I, incumbent):
        if k > incumbent:
            return 1, incumbent
        if P == 0:
            return 1, min(k, incumbent)
        available = P & ~I
        selected  = I
        while selected:
            bit = selected & -selected
            selected -= bit
            available &= ~adj[bit.bit_length() - 1]
        candidates = available
        if canonical and I:
            candidates &= ~((1 << I.bit_length()) - 1)
        total = 1
        if candidates:
            while candidates:
                bit = candidates & -candidates
                candidates -= bit
                count, incumbent = visit(P, k, I | bit, incumbent)
                total += count
        elif I and available == 0:
            count, incumbent = visit(P & ~I, k + 1, 0, incumbent)
            total += count
        return total, incumbent
    return visit((1 << n) - 1, 0, 0, n + 1)

# ---------------------------------------------------------------------------
# Table 4 driver
# ---------------------------------------------------------------------------
def run_table4(instances):
    """instances: list of (label, V, E). Runs the canonical ablation for each."""
    rows = []
    for label, V, E in instances:
        for canonical, variant in [(True, 'canonical'), (False, 'noncanonical')]:
            key = (label, variant)
            t0 = time.perf_counter()
            count, best = independent_count(V, E, canonical)
            indep_s = time.perf_counter() - t0
            row = {
                'instance': label,
                'variant': variant,
                'vertices': len(V),
                'edges': len(E) // 2,
                'paper_popped_occurrences': TABLE4_PAPER.get(key),
                'independent_popped_occurrences': count,
                'best_coloring_size': best,
                'expected_reproduction_count': TABLE4_EXPECTED[key],
                'count_matches_reproduction': count == TABLE4_EXPECTED[key],
                'count_matches_paper': count == TABLE4_PAPER.get(key),
                'independent_runtime_seconds': round(indep_s, 6),
            }
            if key in SKIP_ENGINE:
                row['engine_popped_occurrences'] = None
                row['engine_best_coloring_size'] = None
                row['engine_status'] = 'SKIPPED_INFEASIBLE'
                row['independent_engine_agree'] = None
                row['engine_runtime_seconds'] = None
                row['skip_reason'] = (
                    f'Engine would have to materialise {count:,} states; '
                    f'memoised counter computes the same quantity without '
                    f'materialisation. Counter validated on the canonical case, '
                    f'where both agree at {TABLE4_EXPECTED[(label, "canonical")]:,}.')
                print(f'  {label:<15} {variant:<12} counter={count:>22,}  '
                      f'best={best}  [engine skipped]')
            else:
                t0 = time.perf_counter()
                engine = run_engine(V, E, target=None, timeout=float('inf'),
                                    canonical=canonical)
                eng_s = time.perf_counter() - t0
                agreement = (engine['explored_popped_occurrences'] == count
                             and engine['best_found'] == best
                             and engine['search_exhausted'])
                row.update({
                    'engine_popped_occurrences': engine['explored_popped_occurrences'],
                    'engine_best_coloring_size': engine['best_found'],
                    'engine_status': engine['status'],
                    'independent_engine_agree': agreement,
                    'engine_runtime_seconds': round(eng_s, 6),
                })
                if not agreement:
                    raise RuntimeError(
                        f'Counter and engine disagree for {key}: '
                        f'{count} vs {engine}')
                print(f'  {label:<15} {variant:<12} counter={count:>22,}  '
                      f'engine={engine["explored_popped_occurrences"]:>22,}  '
                      f'best={best}  [both methods agree]')
            rows.append(row)
    return rows

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def write_csv(path, rows):
    if not rows:
        return
    fieldnames = []                      # union of keys, first-seen order
    for r in rows:
        for k in r:
            if k not in fieldnames:
                fieldnames.append(k)
    with path.open('w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=fieldnames, restval='')
        w.writeheader()
        w.writerows(rows)

def ordered_benchmark_files(found_set):
    return [name
            for group in (SMALL_NAMES, MEDIUM_NAMES, LARGE_NAMES)
            for name in sorted(group & found_set)]

# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------
def main():
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except ImportError:
        print('Colab not detected; assuming Drive path is already accessible.')
    if not DIMACS_DIR.exists():
        raise FileNotFoundError(f'Benchmark folder not found: {DIMACS_DIR}')
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    found_set = {n for n in DIMACS_TARGETS if (DIMACS_DIR / n).is_file()}
    ordered   = ordered_benchmark_files(found_set)
    missing   = sorted(set(DIMACS_TARGETS) - found_set)
    assert len(ordered) == len(set(ordered)), 'Benchmark groups overlap'
    print(f'Found {len(ordered)}/{len(DIMACS_TARGETS)} configured DIMACS files; '
          f'timeout={TIMEOUT_SECONDS}s each.')
    if missing:
        print('Missing:', ', '.join(missing))

    # ----- Table 3 ---------------------------------------------------------
    print('\n=== TABLE 3 — DIMACS benchmark ===')
    table3 = []
    for name in ordered:
        V, E, sha, conv, hm, pm = parse_file(DIMACS_DIR / name)
        result = run_engine(V, E, DIMACS_TARGETS[name], TIMEOUT_SECONDS,
                            canonical=True)
        row = {
            'instance': name,
            'vertices': len(V),
            'edges': len(E) // 2,
            'target_chi': DIMACS_TARGETS[name],
            **result,
            'header_edge_count': hm,
            'header_edge_convention': conv,
            'parsed_undirected_edges': pm,
            'input_sha256': sha,
        }
        table3.append(row)
        note = ' [doubled header]' if conv == 'doubled' else ''
        print(f'{name:<18} V={len(V):>3} E={len(E)//2:>5} '
              f'target={row["target_chi"]:>2} '
              f'best={str(row["best_found"]):>4} '
              f'popped={row["explored_popped_occurrences"]:>9,} '
              f'{row["runtime_seconds"]:>7.3f}s {row["status"]}{note}')

    write_csv(OUT_DIR / 'table3_dimacs.csv', table3)   # save early

    # ----- Table 4 ---------------------------------------------------------
    print('\n=== TABLE 4 — canonical-ordering ablation ===')
    table4_instances = []
    # myciel3 from embedded text (parse_dimacs_text returns FIVE values).
    V3, E3, conv3, _hm3, _pm3 = parse_dimacs_text(MYCIEL3_TEXT)
    if conv3 != 'undirected':
        raise RuntimeError(f'Embedded myciel3 header unexpected: {conv3}')
    table4_instances.append(('myciel3.col', V3, E3))
    # myciel4 from DIMACS_DIR (parse_file returns SIX values).
    myciel4_path = DIMACS_DIR / 'myciel4.col'
    if myciel4_path.is_file():
        V4, E4, _sha4, _conv4, _hm4, _pm4 = parse_file(myciel4_path)
        table4_instances.append(('myciel4.col', V4, E4))
    else:
        print('WARNING: myciel4.col not found; Table 4 will only report myciel3.')
    table4 = run_table4(table4_instances)

    # ----- Write CSV and JSON ---------------------------------------------
    write_csv(OUT_DIR / 'table3_dimacs.csv', table3)
    write_csv(OUT_DIR / 'table4_canonical_ablation.csv', table4)

    # ----- Write Markdown -------------------------------------------------
    t3_md = [
        '# Table 3 — DIMACS benchmark results', '',
        f'Per-instance timeout: {TIMEOUT_SECONDS:.1f} s.',
        'Popped occurrences include repeated occurrences, dead ends and pruned nodes.',
        'A target reached before timeout is not an exhaustive proof.', '',
        '| Instance | |V| | |E| | Header edges | Target χ | Best found | '
        'Popped occurrences | Runtime (s) | Target reached | Exhaustive | Status |',
        '|---|---:|---:|---|---:|---:|---:|---:|---|---|---|',
    ]
    for r in table3:
        t3_md.append(
            f'| {r["instance"]} | {r["vertices"]} | {r["edges"]} | '
            f'{r["header_edge_count"]} ({r["header_edge_convention"]}) | '
            f'{r["target_chi"]} | '
            f'{r["best_found"] if r["best_found"] is not None else "N/A"} | '
            f'{r["explored_popped_occurrences"]} | {r["runtime_seconds"]:.3f} | '
            f'{r["target_reached"]} | {r["search_exhausted"]} | {r["status"]} |')
    (OUT_DIR / 'table3_dimacs.md').write_text('\n'.join(t3_md) + '\n', encoding='utf-8')

    t4_md = [
        '# Table 4 — canonical-ordering ablation', '',
        'Metric: popped DFS occurrences, including repeats, dead ends and pruned nodes.',
        'All runs use `timeout=inf` (exhaustive). The noncanonical myciel4 count '
        'is computed by the memoised counter only; see notes below.', '',
        '| Instance | Variant | Counter | Engine | Best χ | Match | Paper |',
        '|---|---|---:|---:|---:|---|---:|',
    ]
    for r in table4:
        engine_cell = ('(skipped)' if r['engine_popped_occurrences'] is None
                       else f'{r["engine_popped_occurrences"]:,}')
        match_cell = ('n/a' if r['independent_engine_agree'] is None
                      else str(r['independent_engine_agree']))
        paper_cell = ('—' if r['paper_popped_occurrences'] is None
                      else f'{r["paper_popped_occurrences"]:,}')
        t4_md.append(
            f'| {r["instance"]} | {r["variant"]} | '
            f'{r["independent_popped_occurrences"]:,} | {engine_cell} | '
            f'{r["best_coloring_size"]} | {match_cell} | {paper_cell} |')
    t4_md += [
        '',
        'The canonical-ordering constraint reduces the popped-occurrence count '
        'of myciel3 from 969,712 to 2,325 (99.76%) and of myciel4 from '
        '3,450,514,238,574,088 to 929,315 (nine orders of magnitude).',
        '',
        'Notes:',
        '- myciel3: both the notebook engine and the memoised counter reproduce '
        '969,712 / 2,325. The paper\'s 5,394 figure for the noncanonical case '
        'was not reproduced by either method; its provenance is unresolved.',
        '- myciel4: both methods reproduce 929,315 canonical. The noncanonical '
        'count of 3,450,514,238,574,088 is obtained by the memoised counter, '
        'because the direct engine cannot materialise that many states in '
        'memory. The two methods agree on every count that both can compute, '
        'which validates the counter for the larger case.',
    ]
    (OUT_DIR / 'table4_canonical_ablation.md').write_text(
        '\n'.join(t4_md) + '\n', encoding='utf-8')

    # ----- Write JSON -----------------------------------------------------
    metadata = {
        'created_utc': datetime.now(timezone.utc).isoformat(),
        'python_version': platform.python_version(),
        'platform': platform.platform(),
        'dimacs_directory': str(DIMACS_DIR),
        'timeout_seconds_per_table3_instance': TIMEOUT_SECONDS,
        'table3_instances_found': len(ordered),
        'table3_instances_missing': missing,
        'benchmark_groups': {
            'small':  sorted(SMALL_NAMES),
            'medium': sorted(MEDIUM_NAMES),
            'large':  sorted(LARGE_NAMES),
        },
        'table3': table3,
        'table4': table4,
        'notebook_git_blob': NOTEBOOK_GIT_BLOB,
        'graph_url': GRAPH_URL,
        'interpretation_note':
            'A timed-out run that reaches the reference target has found a '
            'coloring of that size; it has not proved optimality. Table 4 is '
            'exhaustive (timeout disabled); the noncanonical myciel4 count is '
            'computed by the memoised counter, which is validated against the '
            'direct engine on the canonical case.',
    }
    (OUT_DIR / 'pcir_reproduction_results.json').write_text(
        json.dumps(metadata, indent=2) + '\n', encoding='utf-8')

    # ----- Summary --------------------------------------------------------
    print('\n=== SUMMARY ===')
    print('Table 4, two-row format for the paper:')
    print(f'  {"Instance":<15} {"Without Canonical":>22} {"With Canonical":>15} {"Reduction":>12}')
    for label, _, _ in table4_instances:
        nc = next(r for r in table4
                  if r['instance'] == label and r['variant'] == 'noncanonical')
        c  = next(r for r in table4
                  if r['instance'] == label and r['variant'] == 'canonical')
        nc_v = nc['independent_popped_occurrences']
        c_v  = c ['independent_popped_occurrences']
        red  = (nc_v - c_v) / nc_v * 100
        print(f'  {label:<15} {nc_v:>22,} {c_v:>15,} {red:>11.4f}%')

    print('\nReports saved to', OUT_DIR)
    for n in ['table3_dimacs.csv', 'table3_dimacs.md',
              'table4_canonical_ablation.csv', 'table4_canonical_ablation.md',
              'pcir_reproduction_results.json']:
        print(' -', OUT_DIR / n)
    print('\nNote: Table 3 timed-out rows are machine-dependent throughput '
          'counts; completed rows are deterministic.')

if __name__ == '__main__':
    main()

Mounted at /content/drive
Found 14/14 configured DIMACS files; timeout=30.0s each.

=== TABLE 3 — DIMACS benchmark ===
myciel3.col        V= 11 E=   20 target= 4 best=   4 popped=    2,325   0.014s TARGET_REACHED_EXHAUSTIVE
myciel4.col        V= 23 E=   71 target= 5 best=   5 popped=  929,315   4.579s TARGET_REACHED_EXHAUSTIVE
queen5_5.col       V= 25 E=  160 target= 5 best=   5 popped=5,687,431  30.000s TARGET_REACHED_TIMEOUT [doubled header]
le450_5a.col       V=450 E= 5714 target= 5 best=  13 popped=2,677,773  30.000s TIMEOUT_TARGET_NOT_REACHED
le450_5b.col       V=450 E= 5734 target= 5 best=  13 popped=2,708,939  30.000s TIMEOUT_TARGET_NOT_REACHED
myciel5.col        V= 47 E=  236 target= 6 best=   6 popped=3,323,263  30.000s TARGET_REACHED_TIMEOUT
queen6_6.col       V= 36 E=  290 target= 7 best=   9 popped=7,897,781  30.000s TIMEOUT_TARGET_NOT_REACHED [doubled header]
queen7_7.col       V= 49 E=  476 target= 7 best=  10 popped=7,409,620  30.000s TIMEOUT_TARGET_NOT_REACHED [doubled 